# EduForge - Fine-tuning Gemma-2b-it with Unsloth
This notebook fine-tunes the `unsloth/gemma-2b-it` model on the provided `train.jsonl` dataset.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install matplotlib seaborn datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

# Load the local dataset file
# Make sure train.jsonl is uploaded to your Colab workspace
dataset = load_dataset("json", data_files="train.jsonl", split="train")

# Format it into the 'text' column mapping as required
def format_prompts(examples):
    return {"text": examples["text"]}

dataset = dataset.map(format_prompts, batched=True)
print("Sample prompt:")
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 1, # batch size 1 as required
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # 3 epochs as required
        learning_rate = 2e-4, # learning rate 2e-4 as required
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## Results
Automatically generates and displays a Training Loss plot using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract loss history from trainer state
loss_history = [log["loss"] for log in trainer.state.log_history if "loss" in log]
step_history = [log["step"] for log in trainer.state.log_history if "loss" in log]

# Plot using seaborn
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(x=step_history, y=loss_history, marker="o", color="b")
plt.title("Training Loss Over Time")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

## OpenEnv Integration
Connects the trained model back to the OpenEnv `reset`/`step` logic as defined in the `openenv.yaml` environment manifest to demonstrate the agent's new behavior.

In [ ]:
# Setup model for faster inference
FastLanguageModel.for_inference(model)

import re
import sys
import os
import numpy as np

# Connect to OpenEnv logic (src.environment.openenv_wrapper:EduForgeEnv)
# Assuming the local 'src' directory has been uploaded to the Colab environment

try:
    from src.environment.openenv_wrapper import EduForgeEnv
    
    # Initialize the environment based on openenv.yaml manifest
    env = EduForgeEnv()
    
    # OpenEnv logic loop
    obs, info = env.reset()
    # State might be multidimensional, let's assume it contains confusion or we can mock it
    # Based on train.jsonl, State usually shows Confusion value.
    confusion = obs[0] if isinstance(obs, (list, tuple, np.ndarray)) else obs
    print(f"[OpenEnv] Initial Reset -> State Confusion: {confusion:.4f}")
    
    action_map = {"explain": 0, "worked_example": 1, "analogize": 2, "question": 3, "correct_fact": 4}
    
    for step in range(5):
        # Format prompt to match train.jsonl structure
        prompt = f"### State\nConfusion: {confusion:.4f}\nAction:"
        
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=20, use_cache=True)
        response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        # Extract the action from the response
        action_match = re.search(r"Action:\s*(\w+)", response)
        action_str = action_match.group(1) if action_match else "explain"
        
        action_idx = action_map.get(action_str, 0)
        print(f"\nStep {step+1}: Agent Policy Chose Action '{action_str}'")
        
        # Step the environment
        next_obs, reward, terminated, truncated, info = env.step(action_idx)
        
        confusion = next_obs[0] if isinstance(next_obs, (list, tuple, np.ndarray)) else next_obs
        print(f"[OpenEnv] Step Result -> Reward: {reward:.4f}, Next Confusion: {confusion:.4f}")
        
        if terminated or truncated:
            print("Episode finished.")
            break
            
except ImportError:
    print("Warning: Could not import EduForgeEnv. Ensure your local 'src' directory is in the Colab working path.")
    print("\n--- Simulated OpenEnv Interaction ---")
    
    # Simulate OpenEnv interaction logic if module is missing in Colab
    confusion = 7.15
    print(f"[Simulated OpenEnv] Initial Reset -> State Confusion: {confusion:.4f}")
    
    for step in range(3):
        prompt = f"### State\nConfusion: {confusion:.4f}\nAction:"
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=20, use_cache=True)
        response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        action_match = re.search(r"Action:\s*(\w+)", response)
        action_str = action_match.group(1) if action_match else "explain"
        
        print(f"\nStep {step+1}: Agent Policy Chose Action '{action_str}'")
        
        # Mock environment step
        reward = 1.5 if action_str in ['worked_example', 'explain'] else -0.5
        confusion = max(0.0, confusion - 1.2)
        print(f"[Simulated OpenEnv] Step Result -> Reward: {reward:.4f}, Next Confusion: {confusion:.4f}")
